# 1º Workshop Eixo de Inovação — Pequenos Modelos de Linguagem Abertos

**Rede Feminista em IA para a América Latina e o Caribe**

---

### Pré-requisitos
- Python >= 3.12
- `jupyterlab` instalado (para executar este notebook)

**Todo o resto é instalado automaticamente neste notebook.**

### Modelos do workshop
- **Default:** `Qwen/Qwen2.5-1.5B-Instruct`
- **Fallback (hardware limitado):** `Qwen/Qwen2.5-0.5B-Instruct`


In [ ]:
import sys
print(f"Versão do Python em uso: {sys.version}")


## 0. Caminhos segundo nível e hardware

### 0.1 Objetivo final
Aplicação do conteúdo explorado a um caso de uso de cada projeto.

### 0.2 Escolha seu caminho

| Caminho | Seções | Descrição |
|---------|--------|-----------|
| 🟢 Exploração simples | 1 → 2 → 3 → 6 → 8 → 12 | Setup → inferência básica → Ollama → avaliação → aplicação |
| 🟡 Experimentação | 1 → 2 → 3 → 4 → 5 → 8 → 12 | Setup → inferência → eficiência → tokenização → avaliação → aplicação |
| 🔴 Profundidade | 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 | Fluxo completo com fine-tuning |
| ⚙️ Hardware limitado | 1 → 2 → 6 → 7 → 8 → 12 | Setup → Ollama → llama.cpp → avaliação → aplicação |

### 0.3 Escolha segundo seu hardware
- 🖥️ **Somente CPU** → Priorizar seções 6 (Ollama) e 7 (llama.cpp)
- ⚡ **GPU disponível** → Fluxo completo com Hugging Face (3 → 10)
- ❓ **Não sabe o que tem** → Avançar até 1.4 (detecção de hardware)


---
## 1. Setup local (adaptado ao hardware)

### 1.1-1.2 Ambiente e instalação base
A célula a seguir instala todas as dependências necessárias. O modelo Qwen/Qwen2.5-1.5B-Instruct que usamos no notebook é particularmente potente para seu tamanho ("Small") porque foi treinado com uma abordagem multilíngue muito agressiva pela equipe da Alibaba. Oferece suporte oficial e otimizado para mais de 29 idiomas. Idiomas com Suporte Principal (Alta Qualidade)

Esses idiomas são os que o modelo maneja com maior fluência e menor "taxa de fertilidade" (fragmentação), pois formaram o grosso do seu treinamento:
* Asiáticos: Chinês (Mandarim e variantes), Japonês, Coreano, Vietnamita, Tailandês.
* Europeus: Espanhol, Inglês, Francês, Português, Alemão, Italiano, Russo.
* Oriente Médio: Árabe.


In [ ]:
import subprocess
import sys

# Garantir que pip esteja disponível
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except subprocess.CalledProcessError:
    print("⏳ Instalando pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--default-pip"])

pacotes = [
    "torch",
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "trl",
    "evaluate",
    "scikit-learn",
    "sentencepiece",
    "protobuf",
    "requests",
    "huggingface_hub",
    "matplotlib",
]

print("📦 Instalando pacotes necessários (pode demorar alguns minutos)...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + pacotes
)
print("✅ Pacotes instalados corretamente")


In [ ]:
# ═══ Imports e configuração ═══
import torch
import gc
import time
import json
import random
import os
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# ═══ Configuração do modelo ═══
NOME_MODELO_DEFAULT = "Qwen/Qwen2.5-1.5B-Instruct"
NOME_MODELO_FALLBACK = "Qwen/Qwen2.5-0.5B-Instruct"

# Escolha seu modelo segundo seu hardware:
# - NOME_MODELO_DEFAULT: requer ~16GB RAM (recomendado com GPU ou >=16GB RAM)
# - NOME_MODELO_FALLBACK: requer ~8GB RAM (recomendado para CPU / hardware limitado)
nome_modelo = NOME_MODELO_FALLBACK

print(f"📌 Modelo selecionado: {nome_modelo}")
print(f"   (Trocar para {NOME_MODELO_DEFAULT} se tiver >=16GB RAM livres)")


In [ ]:
# ═══ 1.3 Detecção de hardware ═══
import platform

print("=" * 55)
print("🔍 DETECÇÃO DE HARDWARE")
print("=" * 55)

print(f"\n💻 Sistema: {platform.system()} {platform.machine()}")
print(f"   Python: {sys.version.split()[0]}")

# RAM
if platform.system() == "Darwin":
    ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
elif platform.system() == "Linux":
    ram_bytes = 0
    with open("/proc/meminfo") as f:
        for linha in f:
            if "MemTotal" in linha:
                ram_bytes = int(linha.split()[1]) * 1024
                break
else:
    ram_bytes = 0

ram_gb = ram_bytes / (1024**3) if ram_bytes else 0
if ram_gb > 0:
    print(f"   RAM total: {ram_gb:.1f} GB")

# GPU / MPS
tem_cuda = torch.cuda.is_available()
tem_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if tem_cuda:
    nome_gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"\n⚡ GPU detectada: {nome_gpu} ({vram_gb:.1f} GB VRAM)")
elif tem_mps:
    print(f"\n⚡ GPU detectada: 🍎 Apple Silicon (MPS backend)")
else:
    print(f"\n🖥️ Somente CPU disponível")

# Recomendação
print("\n" + "=" * 55)
if tem_cuda or tem_mps:
    print("✅ Recomendação: Fluxo completo com Hugging Face (caminho 🔴)")
elif ram_gb >= 8:
    print("✅ Recomendação: Você pode usar Hugging Face em CPU (caminho 🟡)")
    print("   Também pode usar Ollama/llama.cpp para mais velocidade")
else:
    print("⚠️ Recomendação: Usar Ollama ou llama.cpp (caminho ⚙️)")
print("=" * 55)


In [ ]:
# ═══ 1.4 Setup alternativo: Ollama e llama.cpp ═══

import platform
import shutil

sistema = platform.system()  # 'Darwin', 'Linux', 'Windows'
print(f"🖥️  Sistema operacional detectado: {sistema}\n")

# ── Ollama ────────────────────────────────────────────────
print("📦 Verificando Ollama...")
if shutil.which("ollama") is not None:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"   ✅ Ollama já instalado: {result.stdout.strip()}")
else:
    print("   ⏳ Instalando Ollama...")
    try:
        if sistema == "Darwin":
            subprocess.check_call(["brew", "install", "ollama"])
        elif sistema == "Linux":
            subprocess.check_call(
                "curl -fsSL https://ollama.ai/install.sh | sh",
                shell=True,
            )
        elif sistema == "Windows":
            subprocess.check_call(
                ["winget", "install", "--id", "Ollama.Ollama", "-e", "--silent"]
            )
        else:
            print(f"   ⚠️  Sistema '{sistema}' não reconhecido.")
            print("      Instalar manualmente em: https://ollama.ai/download")
        if shutil.which("ollama") is not None:
            print("   ✅ Ollama instalado corretamente.")
    except FileNotFoundError as e:
        print(f"   ⚠️  Comando não encontrado: {e}")
        print("      Instalar manualmente em: https://ollama.ai/download")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Erro durante a instalação do Ollama: {e}")

# ── llama-cpp-python ──────────────────────────────────────
print("\n📦 Instalando llama-cpp-python...")
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"]
    )
    print("   ✅ llama-cpp-python instalado corretamente.")
    print("   ℹ️  Requer compilador C++ (cmake/clang). No macOS geralmente funciona sem configuração extra.")
except subprocess.CalledProcessError as e:
    print(f"   ❌ Erro ao instalar llama-cpp-python: {e}")
    print("      No Linux: sudo apt install cmake build-essential")
    print("      No Windows: instalar Visual Studio Build Tools")


---
## 2. Small Open Language Models (contexto mínimo)

### 2.1 O que significa "small"
Um **Small Language Model (SLM)** tem tipicamente entre **0,5B e 7B parâmetros**, em contraste com os LLMs como GPT-4 (estimado >1T parâmetros) ou LLaMA-70B.

Esses modelos são suficientemente capazes para muitas tarefas e suficientemente leves para rodar **localmente** em hardware acessível.

### 2.2 Tradeoffs reais

| Aspecto | SLM (1-3B) | LLM (70B+) |
|---------|-----------|------------|
| RAM necessária (inferência) | 2-6 GB | >140 GB |
| Velocidade (CPU) | Segundos | Minutos |
| Qualidade geral | Boa para tarefas específicas | Excelente generalista |
| Custo de deploy | Baixo / grátis | Alto (GPU necessária) |
| Privacidade | Total (local) | Não garantida (provedor) |
| Fine-tuning | Viável no laptop | Requer cluster |

### 2.3 Casos de uso dos SLM
- **Classificação de textos**
- **Resumo automático**
- **Extração de informações**
- **Assistentes especializados**
- **Tradução / adaptação linguística**

> 🧪 **Experimente isso:** Pense no seu projeto — que tarefa específica poderia fazer um SLM?


---
## 3. Inferência base com Hugging Face

### 3.1 Carregar modelo
Há duas formas principais:
- **Pipeline:** interface de alto nível, rápida de usar
- **Manual:** mais controle sobre tokenização e geração


In [ ]:
# ═══ 3.1 Carregar modelo e tokenizador ═══
print(f"⏳ Carregando modelo: {nome_modelo}")
print("   (na primeira vez baixa os pesos, pode demorar vários minutos)")

tokenizador = AutoTokenizer.from_pretrained(nome_modelo)
if tokenizador.pad_token is None:
    tokenizador.pad_token = tokenizador.eos_token

try:
    modelo = AutoModelForCausalLM.from_pretrained(
        nome_modelo, torch_dtype="auto", device_map="auto"
    )
except Exception as erro_carregamento:
    print(f"⚠️ device_map='auto' falhou ({erro_carregamento}), carregando em CPU...")
    modelo = AutoModelForCausalLM.from_pretrained(
        nome_modelo, torch_dtype=torch.float32
    )

dispositivo = next(modelo.parameters()).device
print(f"\n✅ Modelo carregado em: {dispositivo}")
print(f"   Parâmetros: {sum(p.numel() for p in modelo.parameters()) / 1e6:.0f}M")
print(f"   Dtype: {next(modelo.parameters()).dtype}")


In [ ]:
# ═══ 3.1b Inferência com pipeline ═══
gerador = pipeline("text-generation", model=modelo, tokenizer=tokenizador)

mensagens = [{"role": "user", "content": "Explique brevemente o que é inteligência artificial"}]
resposta = gerador(mensagens)

print("🤖 Resposta (pipeline):")
print(resposta[0]["generated_text"][-1]["content"])


In [ ]:
# ═══ 3.2 Geração manual com diferentes parâmetros ═══

def gerar_resposta(prompt, max_tokens=100, temperatura=0.0, modelo_usar=None):
    """Gera uma resposta dado um prompt ou uma lista de prompts (batching)."""
    _modelo = modelo_usar if modelo_usar is not None else modelo
    _dispositivo = next(_modelo.parameters()).device

    # Suporte para um prompt individual ou uma lista (batching)
    eh_lista = isinstance(prompt, list)
    prompts = prompt if eh_lista else [prompt]

    # Construir lista de conversas (lista de listas de mensagens)
    conversas = [[{"role": "user", "content": p}] for p in prompts]

    # apply_chat_template com tokenize=True aplica o template e tokeniza em um único passo
    # Modelos causais requerem padding à esquerda
    entrada = tokenizador.apply_chat_template(
        conversas,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        padding=True,
        return_dict=True,
    )
    entrada = {k: v.to(_dispositivo) for k, v in entrada.items()}

    params_gen = {
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizador.pad_token_id,
    }
    if temperatura > 0:
        params_gen["do_sample"] = True
        params_gen["temperature"] = temperatura
    else:
        params_gen["do_sample"] = False

    with torch.no_grad():
        saida = _modelo.generate(**entrada, **params_gen)

    # Decodificar somente os tokens novos (sem o prompt de entrada) usando batch_decode
    input_len = entrada["input_ids"].shape[1]
    respostas = tokenizador.batch_decode(saida[:, input_len:], skip_special_tokens=True)

    return respostas if eh_lista else respostas[0]


# Testar com diferentes temperaturas
prompt_exemplo = "Crie um poema sobre as abelhas:"

configuracoes = {
    "Determinístico (temp=0)": 0.0,
    "Determinístico (temp=0) - segunda tentativa, deve ser idêntico ao primeiro": 0.0,
    "Conservador (temp=0.3)": 0.3,
    "Equilibrado (temp=0.7)": 0.7,
    "Criativo (temp=1.2)": 1.2,
}

for nome_config, temp in configuracoes.items():
    print(f"\n{'='*60}")
    print(f"📝 {nome_config}")
    print(f"{'='*60}")
    resposta = gerar_resposta(prompt_exemplo, max_tokens=80, temperatura=temp)
    print(resposta[:300])


### 3.3 Primeiro exercício
🧪 **Experimente isso:** Modifique o prompt para que se relacione com seu projeto.


In [ ]:
# ═══ 3.3 Exercício: seu primeiro prompt aplicado ═══

# 🧪 Modifique este prompt para seu projeto:
meu_prompt = "Quais são as principais aplicações da inteligência artificial para a preservação cultural?"

resposta = gerar_resposta(meu_prompt, max_tokens=150)
print("🤖 Resposta:")
print(resposta)


In [ ]:
# ═══ 3.4 Indo além: batching e streaming ═══

# --- Batching: processar múltiplos prompts em uma única inferência ---
prompts_batch = [
    "O que é aprendizado de máquina?",
    "Quais são os tipos de redes neurais?",
    "O que são dados abertos?",
]

print("📦 Processando batch de prompts:\n")
respostas_det = gerar_resposta(prompts_batch, max_tokens=60)
for i, (prompt, resposta) in enumerate(zip(prompts_batch, respostas_det)):
    print(f"📝 Prompt {i+1}: {prompt}")
    print(f"🤖 {resposta[:200]}")
    print()

# --- Streaming ---
from transformers import TextStreamer

print("\n--- Streaming (a resposta aparece token por token) ---\n")
streamer = TextStreamer(tokenizador, skip_special_tokens=True, skip_prompt=True)

mensagens = [{"role": "user", "content": "Explique brevemente o que é machine learning."}]
texto_chat = tokenizador.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
entrada = tokenizador(texto_chat, return_tensors="pt")
_dispositivo = next(modelo.parameters()).device
entrada = {k: v.to(_dispositivo) for k, v in entrada.items()}

print("🤖 Resposta (streaming):")
with torch.no_grad():
    modelo.generate(
        **entrada, max_new_tokens=80,
        streamer=streamer, pad_token_id=tokenizador.pad_token_id,
    )


---
## 4. Inferência eficiente (Hugging Face)

### 4.1 Quantização (8-bit / 4-bit)
A **quantização** reduz a precisão numérica dos pesos do modelo para economizar memória e acelerar a inferência.
A redução não é linear porque apenas as camadas lineares são comprimidas, enquanto as camadas de embedding, normalização e cabeçalhos (heads) permanecem em alta precisão (16/32-bit) para evitar perda de inteligência do modelo.

| Precisão | Bytes/param | RAM (0.5B) | Qualidade |
|----------|------------|-----------|---------|
| float32 | 4 | ~1.84 GB | Máxima |
| float16/bf16 | 2 | ~0.92 GB | Muito boa |
| 8-bit (int8) | 1 | ~0.59 GB | Boa |
| 4-bit (nf4) | 0.5 | ~0.42 GB | Aceitável |

> ⚠️ A quantização com `bitsandbytes` requer GPU NVIDIA (CUDA). Em CPU, as alternativas são Ollama e llama.cpp (seções 6-7) que usam formato GGUF.


In [ ]:
# ═══ 4.1-4.2 Quantização e uso de memória ═══

prompt_teste = "O que é inteligência artificial? Responda em uma frase."

def memoria_modelo(m):
    return m.get_memory_footprint() / (1024**3)

def testar_precisao(rotulo, m):
    dtype = next(m.parameters()).dtype
    mem = memoria_modelo(m)
    resp = gerar_resposta(prompt_teste, max_tokens=50, modelo_usar=m)
    print(f"\n{'='*58}")
    print(f"🔢 {rotulo}")
    print(f"   Dtype:     {dtype}")
    print(f"   Memória:   {mem:.2f} GB")
    print(f"   Resposta: {resp[:150]}")
    return mem

resultados_precisao = {}

# ── float32 (precisão completa) ─────────────────────────
print("⏳ Carregando modelo em float32 (precisão completa)...")
modelo_fp32 = AutoModelForCausalLM.from_pretrained(
    nome_modelo, torch_dtype=torch.float32, device_map="auto"
)
resultados_precisao["float32"] = testar_precisao("float32 (precisão completa)", modelo_fp32)
del modelo_fp32
gc.collect()

# ── float16 (16-bit) ─────────────────────────────────────
print("\n⏳ Carregando modelo em float16 (16-bit)...")
modelo_fp16 = AutoModelForCausalLM.from_pretrained(
    nome_modelo, torch_dtype=torch.float16, device_map="auto"
)
resultados_precisao["float16"] = testar_precisao("float16 (16-bit)", modelo_fp16)
del modelo_fp16
gc.collect()

# ── int8 e nf4 (requerem GPU CUDA + bitsandbytes) ───────
if tem_cuda or tem_mps:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
        from transformers import BitsAndBytesConfig
        torch.cuda.empty_cache()

        # 8-bit (int8)
        print("\n⏳ Carregando modelo em 8-bit (int8)...")
        modelo_8bit = AutoModelForCausalLM.from_pretrained(
            nome_modelo,
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto",
        )
        resultados_precisao["int8"] = testar_precisao("int8 (8-bit)", modelo_8bit)
        del modelo_8bit
        gc.collect()
        torch.cuda.empty_cache()

        # 4-bit (nf4)
        print("\n⏳ Carregando modelo em 4-bit (nf4)...")
        modelo_4bit = AutoModelForCausalLM.from_pretrained(
            nome_modelo,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            ),
            device_map="auto",
        )
        resultados_precisao["nf4"] = testar_precisao("nf4 (4-bit)", modelo_4bit)
        del modelo_4bit
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n⚠️ Erro com quantização bitsandbytes: {e}")
else:
    print("\n⚠️ Quantização 8-bit e 4-bit requerem GPU NVIDIA (CUDA).")
    print("   Em CPU/MPS somente float32 e float16 estão disponíveis.")
    print("   Alternativas para CPU: Ollama e llama.cpp (seções 6-7).")

# ── Resumo comparativo ───────────────────────────────────
if resultados_precisao:
    mem_base = resultados_precisao.get("float32", next(iter(resultados_precisao.values())))
    print(f"\n{'='*58}")
    print("📊 Resumo comparativo de memória:")
    print(f"{'Precisão':<12}  {'Memória (GB)':>12}  {'Redução':>10}")
    print("-" * 38)
    for nome, mem in resultados_precisao.items():
        reducao = (1 - mem / mem_base) * 100
        print(f"{nome:<12}  {mem:>12.2f}  {reducao:>9.0f}%")


In [ ]:
# ═══ 4.3-4.4 Velocidade de inferência ═══

prompt_velocidade = "Explique brevemente o que é deep learning."
n_repeticoes = 10
max_tokens_velocidade = 50

def medir_velocidade(m, rotulo):
    tempos = []
    for i in range(n_repeticoes):
        inicio = time.time()
        _ = gerar_resposta(prompt_velocidade, max_tokens=max_tokens_velocidade, modelo_usar=m)
        tempos.append(time.time() - inicio)
    media = np.mean(tempos)
    tps = max_tokens_velocidade / media
    dispositivo = next(m.parameters()).device
    dtype = next(m.parameters()).dtype
    print(f"\n{'='*52}")
    print(f"⚡ {rotulo}")
    print(f"   Dispositivo: {dispositivo}  |  Dtype: {dtype}")
    print(f"   Tempo médio: {media:.2f}s (±{np.std(tempos):.2f}s)")
    print(f"   Tokens/segundo estimado: ~{tps:.1f}")
    return media, tps

resultados_velocidade = {}

# ── Dispositivo atual (GPU/MPS se disponível) ───────────
print("⏱️ Medindo velocidade de inferência...\n")
disp_atual = str(next(modelo.parameters()).device).split(":")[0]
t_acelerador, tps_acelerador = medir_velocidade(modelo, f"Acelerador ({disp_atual})")
resultados_velocidade[disp_atual] = (t_acelerador, tps_acelerador)

# ── Comparação com CPU (somente se houver GPU ou MPS) ────
if tem_cuda or tem_mps:
    print("\n⏳ Carregando modelo em CPU para comparação...")
    modelo_cpu = AutoModelForCausalLM.from_pretrained(
        nome_modelo,
        torch_dtype=next(modelo.parameters()).dtype,
        device_map="cpu",
    )
    t_cpu, tps_cpu = medir_velocidade(modelo_cpu, "CPU")
    resultados_velocidade["cpu"] = (t_cpu, tps_cpu)
    del modelo_cpu
    gc.collect()

    print(f"\n{'='*52}")
    print("📊 Resumo comparativo:")
    print(f"{'Dispositivo':<14}  {'Média (s)':>13}  {'Tok/s':>7}  {'Aceleração':>12}")
    print("-" * 52)
    for disp, (t, tps) in resultados_velocidade.items():
        aceleracao = t_cpu / t if disp != "cpu" else 1.0
        print(f"{disp:<14}  {t:>13.2f}  {tps:>7.1f}  {aceleracao:>11.1f}x")
else:
    print(f"\n{'='*52}")
    print("📊 Somente CPU disponível — sem comparação de aceleração.")


---
## 5. Tokenização, comprimento e fertility rate

### 5.1 O que é um token
Um **token** é a unidade mínima que um modelo de linguagem processa. Não é necessariamente uma palavra: pode ser uma subpalavra, um caractere, ou até mesmo um byte.

**Por que importa?**
- O **custo** de usar um modelo é medido em tokens
- A **janela de contexto** tem um limite fixo de tokens
- Um mesmo texto em idiomas diferentes pode ter quantidades muito diferentes de tokens

O vocabulário do Qwen2.5 tem 151.643 tokens. É enorme comparado com modelos antigos (como o LLaMA 2 que tinha 32k).
* A vantagem: Isso permite que idiomas como o Chinês ou o Árabe tenham tokens dedicados para palavras inteiras, reduzindo sua fertilidade.
* A desvantagem: Se um idioma (como o Guarani) não está representado nesses 151k tokens, o modelo "quebra" e se torna muito ineficiente.


In [ ]:
# ═══ 5.1-5.2 Como o modelo tokeniza ═══

texto_exemplo = "A inteligência artificial pode transformar a sociedade"

tokens_ids = tokenizador.encode(texto_exemplo)
tokens_texto = tokenizador.tokenize(texto_exemplo)

print(f"📝 Texto: '{texto_exemplo}'")
print(f"   Palavras: {len(texto_exemplo.split())}")
print(f"   Tokens: {len(tokens_ids)}")
print(f"   Tokens (texto): {tokens_texto}")
print()

# Decodificar token por token
print("🔍 Decodificação token por token:")
for tid in tokens_ids:
    fragmento = tokenizador.decode([tid])
    print(f"   ID {tid:6d} → '{fragmento}'")


In [ ]:
# ═══ 5.3 Tokenização desigual entre idiomas ═══

frases = {
    "Portugues": "A inteligência artificial pode transformar a sociedade",
    "Ingles":    "Artificial intelligence can transform society",
    "Frances":   "L'intelligence artificielle peut transformer la société",
    "Mandarin":  "人工智能可以改变社会",
    "Tailandes": "ปัญญาประดิษฐ์สามารถเปลี่ยนแปลงสังคมได้",
    "Quechua":   "Kapchisqa yachayqa llaqtata tikranman",
    "Guarani":   "Ava japopyre arandu ikatu omoambue avano'õme",
    "Arabe":     "يمكن للذكاء الاصطناعي أن يغير المجتمع",
    "Hebreo":    "בינה מלאכותית יכולה לשנות את החברה",
}

print("📊 Comparação de tokenização entre idiomas\n")
print(f"{'Idioma':<12} {'Palavras':<10} {'Tokens':<10} {'Ratio':<10}")
print("-" * 42)

for idioma, frase in frases.items():
    tokens = tokenizador.encode(frase)
    palavras = len(frase.split())
    ratio = len(tokens) / max(palavras, 1)
    print(f"{idioma:<12} {palavras:<10} {len(tokens):<10} {ratio:<10.2f}")

print()
print("💡 Um ratio mais alto significa que o modelo precisa de mais tokens para")
print("   representar o mesmo conteúdo → maior custo e menor contexto disponível.")


In [ ]:
# ═══ 5.4-5.5 Fertility Rate ═══

print("📊 Fertility Rate: tokens por unidade linguística\n")

textos_comparacao = {
    "Portugues": [
        "A inteligência artificial transforma a medicina",
        "Os dados abertos permitem transparência",
        "A tecnologia deve ser acessível para todos",
    ],
    "Ingles": [
        "Artificial intelligence transforms medicine today",
        "Open data enables transparency always",
        "Technology must be accessible for everyone",
    ],
    "Quechua": [
        "Yachay rurana atinmi hampiq llamkayta tikray",
        "Kichqa willakuykunaqa sutichasqa kanan",
        "Tecnologia lliw runakunapaqmi kanan",
    ],
}

resultados_fr = {}
for idioma, textos in textos_comparacao.items():
    fertility_rates = []
    for texto in textos:
        tokens = tokenizador.encode(texto)
        palavras = texto.split()
        fr = len(tokens) / max(len(palavras), 1)
        fertility_rates.append(fr)

    media_fr = np.mean(fertility_rates)
    resultados_fr[idioma] = media_fr
    print(f"  {idioma}: fertility rate médio = {media_fr:.2f} tokens/palavra")
    for i, texto in enumerate(textos):
        toks = len(tokenizador.encode(texto))
        pals = len(texto.split())
        print(f"    '{texto[:50]}...' → {pals} palavras, {toks} tokens")
    print()

print("💡 Idiomas com menor representação nos dados de treinamento")
print("   tendem a ter fertility rates mais altos, o que implica:")
print("   • Maior custo computacional pelo mesmo conteúdo")
print("   • Menor janela de contexto efetiva")
print("   • Potencial perda de qualidade nas respostas")



### 5.6 O Risco da Contaminação Semântica (Cross-lingual Interference)

Quando um modelo como o Qwen2.5 vê uma sequência de caracteres que existe tanto em português quanto em um idioma indígena, mas com significados totalmente distintos, ocorre o seguinte:

**Apropriação de Tokens**: O tokenizador atribui um token que "conhece" (do português) a uma cadeia do idioma local. O fertility rate cai (porque usa um único token), mas o significado está corrompido.

**Viés do Vetor (Embedding)**: Por serem modelos treinados majoritariamente em português/inglês que compartilham o alfabeto, o "vetor" desse token está carregado com o significado do idioma dominante.

Um fertility rate baixo pode ser um sintoma de **colonização linguística** no modelo.
* Se o modelo usa poucos tokens para o Guarani porque está "reciclando" tokens do português ou de outro idioma com o mesmo alfabeto, ele não está entendendo o Guarani; está forçando a língua local dentro de uma estrutura semântica alheia.
* Isso é perigoso para os projetos de Busca Semântica: o buscador poderia encontrar documentos irrelevantes baseando-se em similaridades superficiais de escrita (superficial similarity) em vez de conceitos.


### 5.7 Discussão
- Quais são as implicações da tokenização desigual para diferentes idiomas?
- Se um idioma precisa de 3x mais tokens para dizer a mesma coisa, o que acontece com o custo e a qualidade?
- Como isso afeta línguas indígenas ou variedades linguísticas menos representadas?

> 🧪 **Experimente isso:** Adicione frases em outros idiomas ou dialetos e observe a fragmentação.


---
## 6. Inferência local simples com Ollama

**Ollama** simplifica rodar modelos localmente. É ideal para:
- Prototipagem rápida sem escrever código Python complexo
- Rodar modelos quantizados (GGUF) eficientemente em CPU
- APIs locais compatíveis com OpenAI

### 6.1 Setup mínimo manual (não é necessário executar, roda automaticamente na próxima célula)
```bash
ollama serve           # iniciar servidor
ollama pull hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q2_K  # baixar modelo
ollama run hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q2_K   # chat interativo
```


In [ ]:
# ═══ 6.1 Setup do Ollama ═══

import shutil
import time

# ── 1. Verificar instalação ───────────────────────────────
if shutil.which("ollama") is None:
    print("⚠️  Ollama não encontrado. Execute primeiro a célula 1.4 para instalá-lo.")
else:
    print(f"✅ Ollama encontrado em: {shutil.which('ollama')}")

    # ── 2. Iniciar servidor em segundo plano (se não estiver rodando já) ─
    # Forçar uso de CPU no Ollama (ignorar GPU/MPS mesmo que disponíveis)
    import os
    os.environ["OLLAMA_NUM_GPU"] = "0"
    try:
        r = __import__("requests").get("http://localhost:11434/api/tags", timeout=2)
        print("✅ Servidor Ollama já em execução.")
        print("   ℹ️  Se o servidor já estava rodando, reinicie-o para aplicar CPU-only.")
    except Exception:
        print("⏳ Iniciando servidor Ollama em segundo plano (CPU-only)...")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            env=os.environ,
        )
        # Aguardar inicialização
        for _ in range(10):
            time.sleep(1)
            try:
                __import__("requests").get("http://localhost:11434/api/tags", timeout=1)
                print("✅ Servidor Ollama iniciado.")
                break
            except Exception:
                pass
        else:
            print("❌ O servidor não respondeu a tempo. Verifique com: ollama serve")

    # ── 3. Baixar modelo se não estiver disponível ────────────
    MODELO_OLLAMA = "hf.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF:Q4_K_M"
    print(f"\n⏳ Verificando modelo '{MODELO_OLLAMA}'...")
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True, text=True, check=True,
        )
        if MODELO_OLLAMA in result.stdout:
            print(f"   ✅ Modelo '{MODELO_OLLAMA}' já disponível.")
        else:
            print(f"   ⏳ Baixando '{MODELO_OLLAMA}' (pode demorar alguns minutos)...")
            subprocess.check_call(["ollama", "pull", MODELO_OLLAMA])
        print(f"   ✅ Modelo '{MODELO_OLLAMA}' baixado.")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Erro: {e}")


In [ ]:
# ═══ 6.2-6.3 Uso do Ollama a partir do Python ═══
import requests

OLLAMA_URL = "http://localhost:11434"
ollama_disponivel = False

try:
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
    if r.status_code == 200:
        ollama_disponivel = True
        modelos_ollama = r.json().get("models", [])
        print("✅ Ollama disponível. Modelos instalados:")
        for m in modelos_ollama:
            print(f"   - {m['name']}")
except Exception:
    pass

if not ollama_disponivel:
    print("ℹ️  Ollama não está disponível neste ambiente.")
    print("   Para instalar:")
    print("   macOS: brew install ollama")
    print("   Linux: curl -fsSL https://ollama.ai/install.sh | sh")
    print("   Depois: ollama serve && ollama pull qwen2.5:0.5b")


In [ ]:
# ═══ 6.4-6.5 Exercício com Ollama ═══

if ollama_disponivel:
    prompt_ollama = "O que é inteligência artificial?"

    print(f"📝 Prompt: {prompt_ollama}")
    print(f"🔧 Modelo: {MODELO_OLLAMA}\n")

    try:
        resposta_ollama = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODELO_OLLAMA,
                "prompt": prompt_ollama,
                "stream": False,
                "options": {"num_gpu": 0},
            },
            timeout=120,
        )
        if resposta_ollama.status_code == 200:
            texto_ollama = resposta_ollama.json()["response"]
            print(f"🤖 Resposta do Ollama:\n{texto_ollama}")
        else:
            print(f"⚠️ Erro: {resposta_ollama.status_code}")
            print(f"   Certifique-se de ter o modelo baixado: ollama pull {MODELO_OLLAMA}")
    except Exception as e:
        print(f"⚠️ Erro ao conectar com Ollama: {e}")

    # Quando usar Ollama vs Hugging Face:
    print("\n📋 Ollama vs Hugging Face:")
    print("   Ollama: mais simples, modelos quantizados, ideal para CPU")
    print("   HF: mais controle, fine-tuning, melhor para experimentação")

    # ── Comparação de velocidade HF vs Ollama ─────────────────
    try:
        _ = modelo  # verificar que o modelo HF está carregado
        hf_disponivel = True
    except NameError:
        hf_disponivel = False

    if hf_disponivel:
        N_RUNS = 10
        MAX_TOK = 50
        print(f"\n{'='*58}")
        print(f"⚡ Comparação de velocidade HF vs Ollama ({N_RUNS} execuções)")
        print(f"   Prompt: {prompt_ollama}")
        print(f"   Max tokens: {MAX_TOK}")

        # Medir HF — contar tokens reais do output
        tempos_hf = []
        tokens_hf = []
        for i in range(N_RUNS):
            t0 = time.time()
            resp_hf = gerar_resposta(prompt_ollama, max_tokens=MAX_TOK)
            tempos_hf.append(time.time() - t0)
            tokens_hf.append(len(tokenizador.encode(resp_hf)))
        media_hf = np.mean(tempos_hf)
        tps_hf = float(np.mean([tok / t for tok, t in zip(tokens_hf, tempos_hf)]))

        print(f"\n🤗 Hugging Face  ({next(modelo.parameters()).device}, {next(modelo.parameters()).dtype})")
        for i, (t, tok) in enumerate(zip(tempos_hf, tokens_hf), 1):
            print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
        print(f"   Média: {media_hf:.2f}s  |  ~{tps_hf:.1f} tok/s")

        # Medir Ollama — usar eval_count do JSON de resposta
        tempos_ollama = []
        tokens_ollama = []
        for i in range(N_RUNS):
            t0 = time.time()
            r_t = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json={
                    "model": MODELO_OLLAMA,
                    "prompt": prompt_ollama,
                    "stream": False,
                    "options": {"num_predict": MAX_TOK, "num_gpu": 0},
                },
                timeout=120,
            )
            tempos_ollama.append(time.time() - t0)
            tokens_ollama.append(r_t.json().get("eval_count", MAX_TOK))
        media_ollama = np.mean(tempos_ollama)
        tps_ollama = float(np.mean([tok / t for tok, t in zip(tokens_ollama, tempos_ollama)]))

        print(f"\n🦙 Ollama  ({MODELO_OLLAMA})")
        for i, (t, tok) in enumerate(zip(tempos_ollama, tokens_ollama), 1):
            print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
        print(f"   Média: {media_ollama:.2f}s  |  ~{tps_ollama:.1f} tok/s")

        # Resumo
        mais_rapido = "HF" if media_hf < media_ollama else "Ollama"
        ratio_velocidade = max(media_hf, media_ollama) / min(media_hf, media_ollama)
        print(f"\n{'='*58}")
        print(f"{'Motor':<12}  {'Média (s)':>13}  {'Tok/s':>7}")
        print("-" * 36)
        print(f"{'HF':<12}  {media_hf:>13.2f}  {tps_hf:>7.1f}")
        print(f"{'Ollama':<12}  {media_ollama:>13.2f}  {tps_ollama:>7.1f}")
        print(f"\n🏆 {mais_rapido} é {ratio_velocidade:.1f}x mais rápido neste ambiente.")
    else:
        print("\nℹ️  Modelo HF não carregado — sem comparação de velocidade.")
        print("   Esta seção requer ter o Ollama rodando localmente")

else:
    print("⏩ Pulando (Ollama não disponível)")


---
## 7. Inferência otimizada com llama.cpp

### 7.1 O que é GGUF e quantização
**GGUF** é um formato de modelo otimizado para rodar em CPU. Armazena os pesos quantizados de forma eficiente.

**llama.cpp** é uma implementação em C++ que executa modelos GGUF com alta eficiência, usando SIMD, multithreading e outros truques de baixo nível.

### 7.3 Parâmetros de performance
- `n_threads`: número de threads de CPU a usar
- `n_batch`: tamanho do batch de tokens
- `n_ctx`: tamanho da janela de contexto


In [ ]:
# ═══ 7.2-7.5 Inferência com llama.cpp via Python ═══

llama_cpp_disponivel = False
try:
    from llama_cpp import Llama
    llama_cpp_disponivel = True
    print("✅ llama-cpp-python disponível")
except ImportError:
    print("ℹ️  llama-cpp-python não está instalado.")
    print("   Para instalar: pip install llama-cpp-python")
    print("   Nota: requer compilador C++ (cmake, clang)")

if llama_cpp_disponivel:
    # Repo HF equivalente e nome do arquivo
    GGUF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
    GGUF_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

    from huggingface_hub import hf_hub_download
    print(f"\n⏳ Baixando modelo GGUF '{GGUF_FILE}' do HuggingFace...")
    caminho_gguf = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
    print(f"   ✅ Salvo em: {caminho_gguf}")

    # ── Carregar com llama.cpp — CPU only (n_gpu_layers=0) ────
    n_threads = os.cpu_count()
    print(f"\n⏳ Carregando modelo no llama.cpp (CPU only, {n_threads} threads)...")
    llm = Llama(
        model_path=caminho_gguf,
        n_ctx=512,
        n_threads=n_threads,
        n_gpu_layers=0,       # forçar CPU mesmo com GPU/MPS disponível
        verbose=False,
    )
    print("   ✅ Modelo carregado.")

    # ── Inferência de teste ────────────────────────────────────
    prompt_llama = prompt_ollama
    MAX_TOK_LLAMA = 50
    N_RUNS_LLAMA = 10

    print(f"\n📝 Prompt: {prompt_llama}")
    saida_llama = llm(prompt_llama, max_tokens=MAX_TOK_LLAMA, echo=False)
    print(f"🤖 Resposta llama.cpp:\n{saida_llama['choices'][0]['text'].strip()}")

    # ── Benchmark llama.cpp — usar completion_tokens do output ─
    tempos_llama = []
    tokens_llama = []
    for i in range(N_RUNS_LLAMA):
        t0 = time.time()
        out_llama = llm(prompt_llama, max_tokens=MAX_TOK_LLAMA, echo=False)
        tempos_llama.append(time.time() - t0)
        tokens_llama.append(out_llama["usage"]["completion_tokens"])
    media_llama = np.mean(tempos_llama)
    tps_llama = float(np.mean([tok / t for tok, t in zip(tokens_llama, tempos_llama)]))

    print(f"\n⚡ llama.cpp ({N_RUNS_LLAMA} runs, CPU only):")
    for i, (t, tok) in enumerate(zip(tempos_llama, tokens_llama), 1):
        print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
    print(f"   Média: {media_llama:.2f}s  |  ~{tps_llama:.1f} tok/s")

    # ── Comparação com Ollama (se disponível) ────────────────
    if ollama_disponivel:
        print(f"\n🦙 Ollama  ({MODELO_OLLAMA}, CPU only):")
        tempos_ollama_cmp = []
        tokens_ollama_cmp = []
        for i in range(N_RUNS_LLAMA):
            t0 = time.time()
            r_cmp = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json={
                    "model": MODELO_OLLAMA,
                    "prompt": prompt_llama,
                    "stream": False,
                    "options": {"num_predict": MAX_TOK_LLAMA, "num_gpu": 0},
                },
                timeout=120,
            )
            tempos_ollama_cmp.append(time.time() - t0)
            tokens_ollama_cmp.append(r_cmp.json().get("eval_count", MAX_TOK_LLAMA))
        media_ollama_cmp = np.mean(tempos_ollama_cmp)
        tps_ollama_cmp = float(np.mean([tok / t for tok, t in zip(tokens_ollama_cmp, tempos_ollama_cmp)]))

        for i, (t, tok) in enumerate(zip(tempos_ollama_cmp, tokens_ollama_cmp), 1):
            print(f"   Run {i}: {t:.2f}s  ({tok} tokens)")
        print(f"   Média: {media_ollama_cmp:.2f}s  |  ~{tps_ollama_cmp:.1f} tok/s")

        # Resumo
        vencedor = "llama.cpp" if media_llama < media_ollama_cmp else "Ollama"
        fator = max(media_llama, media_ollama_cmp) / min(media_llama, media_ollama_cmp)
        print(f"\n{'='*58}")
        print(f"📊 Comparação CPU — mesmo modelo ({GGUF_FILE})")
        print(f"{'Motor':<14}  {'Média (s)':>13}  {'Tok/s':>7}")
        print("-" * 38)
        print(f"{'llama.cpp':<14}  {media_llama:>13.2f}  {tps_llama:>7.1f}")
        print(f"{'Ollama':<14}  {media_ollama_cmp:>13.2f}  {tps_ollama_cmp:>7.1f}")
        print(f"\n🏆 {vencedor} é {fator:.2f}x mais rápido neste ambiente.")
    else:
        print("\nℹ️  Ollama não disponível — comparação omitida.")
        print(f"   llama.cpp: {media_llama:.2f}s média  |  ~{tps_llama:.1f} tok/s")

    del llm
    gc.collect()

else:
    print("\n⏩ llama-cpp-python não disponível — pulando seção 7.")


---
## 8. Avaliação, reprodutibilidade e alucinações

### 8.1 Avaliação qualitativa
Ao avaliar um modelo, podemos considerar estes aspectos:
- **Utilidade:** ¿respondeu ao que foi perguntado? / **Utilidade:** ¿serve à pessoa que perguntou para atingir seus objetivos?
- **Clareza:** ¿a resposta é compreensível? / **Clareza:** ¿a pessoa que lê a resposta, a compreende?
- **Consistência:** ¿fornece respostas semelhantes a prompts semelhantes? / **Robustez semântica:** ¿o modelo mantém sua postura diante de mudanças superficiais no prompt, ou sua instabilidade revela falta de "raciocínio" sólido?
- **Precisão:** ¿a informação está correta? / **Veracidade:** se a informação é verificável, ¿ela é verdadeira?
- **Neutralidade:** ¿evita vieses ou estereótipos? / **Posicionalidade:** ¿reproduz vieses ou estereótipos sob a perspectiva de qual grupo ou grupos sociais? / **Sycophancy:** ¿o modelo parece tentar alinhar-se com a posicionalidade do usuário? De ser assim, ¿essa tentativa é percebida como caricata ou forçada?


In [ ]:
# ═══ 8.1 Avaliação qualitativa ═══

prompts_avaliacao = [
    "Quais são as vantagens de usar software de código aberto?",
    "Descreva as diferenças entre Python e JavaScript.",
    "Quais são os riscos de depender de uma única ferramenta tecnológica?",
]

print("🔍 Avaliação qualitativa — analisando qualidade das respostas\n")
for prompt in prompts_avaliacao:
    resposta = gerar_resposta(prompt, max_tokens=1000)
    print(f"📝 Prompt: {prompt}")
    print(f"🤖 Resposta: {resposta}")
    print(f"   🧪 A resposta é útil, clara, precisa e neutra? ")
    print("-" * 60)
    print()


### 8.1 Avaliação qualitativa


In [ ]:
# ═══ 8.2 Métricas básicas ═══
from sklearn.metrics import accuracy_score, f1_score

# Simulamos uma tarefa de classificação de sentimento
textos_avaliacao = [
    ("Este curso de programação é excelente e muito completo", "positivo"),
    ("O servidor caiu três vezes esta semana, é frustrante", "negativo"),
    ("O relatório tem dados interessantes sobre o tema", "positivo"),
    ("Nada funcionou do que testamos no laboratório", "negativo"),
    ("Os resultados do experimento são aceitáveis", "positivo"),
    ("Foi terrível a experiência com aquela atualização", "negativo"),
    ("A ferramenta cumpre sua função sem problemas", "positivo"),
    ("O desempenho do sistema é inaceitavelmente lento", "negativo"),
]

prompt_classificacao = (
    "Classifique o seguinte texto como 'positivo' ou 'negativo'. "
    "Responda SOMENTE com uma palavra.\n\nTexto: {texto}\n\nClassificação:"
)

print("📊 Avaliação quantitativa — classificação de sentimento\n")

rotulos_verdadeiros = []
previsoes = []

for texto, rotulo_real in textos_avaliacao:
    resposta = gerar_resposta(
        prompt_classificacao.format(texto=texto), max_tokens=5, temperatura=0.0
    )
    previsao = resposta.strip().lower().rstrip(".")
    # Normalizar previsão
    if "positiv" in previsao:
        previsao_norm = "positivo"
    elif "negativ" in previsao:
        previsao_norm = "negativo"
    else:
        previsao_norm = previsao

    rotulos_verdadeiros.append(rotulo_real)
    previsoes.append(previsao_norm)
    marca = "✅" if previsao_norm == rotulo_real else "❌"
    print(f"  {marca} '{texto[:50]}...' → real: {rotulo_real}, previsto: {previsao_norm}")

# Calcular métricas
exatidao = accuracy_score(rotulos_verdadeiros, previsoes)
f1 = f1_score(rotulos_verdadeiros, previsoes, average="weighted", zero_division=0)

print(f"\n📊 Resultados:")
print(f"   Accuracy: {exatidao:.2%}")
print(f"   F1 (weighted): {f1:.2%}")

# Matriz de confusão
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib import pyplot as plt

cm = confusion_matrix(rotulos_verdadeiros, previsoes, labels=["positivo", "negativo"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["positivo", "negativo"])
disp.plot(cmap="Blues")
plt.title("Matriz de Confusão")
plt.show()


### **8.3 Reprodutibilidade básica**

A reprodutibilidade em modelos de linguagem grandes (LLMs) depende de vários fatores.

* A semente ("seed"): é um valor inicial usado para gerar números aleatórios.
* A temperatura ("temperature"): controla o grau de aleatoriedade na geração de texto mediante a distribuição de probabilidade das próximas palavras ajustando a equação de softmax

$$ \sigma(\mathbf{z})_i = \frac{\exp\left(\frac{z_i}{T}\right)}{\sum_{j=1}^{C} \exp\left(\frac{z_j}{T}\right)} $$

onde $z_i$ é o logit da palavra $i$, $T$ é a temperatura, e $C$ é o número total de palavras no vocabulário.

-> Se usarmos T=1, a distribuição de probabilidade se mantém sem alterações. Se T<1, a distribuição se torna mais "pontiaguda", aumentando a probabilidade das palavras mais prováveis e reduzindo a das menos prováveis, tornando o modelo mais determinístico. No limite quando T→0, o modelo se torna completamente determinístico, sempre escolhendo a palavra com maior probabilidade. Se T>1, a distribuição se torna mais "plana", aumentando a probabilidade das palavras menos prováveis e tornando o modelo mais criativo e diverso.

-> Fixar uma semente ("seed") nas funções de geração ajuda a obter resultados repetíveis, mas só garante saídas idênticas quando a temperatura é 0 (modo determinístico) e o hardware e as versões das bibliotecas são consistentes. Além disso, operações em GPU/MPS podem não ser 100% determinísticas. Por isso, embora fixar sementes e usar temperature=0 melhore a reprodutibilidade, na prática sempre há alguma margem de variação nos resultados dos LLMs.

-> Se usarmos temperature>1, o modelo gradualmente introduz aleatoriedade controlada: com a mesma semente e hardware, as respostas consecutivas podem variar devido à natureza estocástica da amostragem com uma distribuição mais suave.


In [ ]:
# ═══ 8.3-8.4 Reprodutibilidade ═══

def fixar_semente(semente):
    """Fixa todas as sementes para reprodutibilidade."""
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semente)

prompt_repro = "Quais são as vantagens de aprender a programar?"

# Experimento 1: temperature=0 (determinístico)
print("═══ Experimento 1: temperature=0 (determinístico) ═══\n")
respostas_1 = []
for i in range(3):
    resp = gerar_resposta(prompt_repro, max_tokens=60, temperatura=0.0)
    respostas_1.append(resp)
    print(f"  Tentativa {i+1}: {resp}")

iguais = all(r == respostas_1[0] for r in respostas_1)
print(f"\n  Todas idênticas? {'✅ Sim' if iguais else '❌ Não'}")

# Experimento 2: temperature=1, fixando semente antes de cada iteração
print("\n═══ Experimento 2: temperature=1, mesma semente antes de cada iteração  ═══\n")
respostas_2 = []
for i in range(3):
    fixar_semente(42)
    resp = gerar_resposta(prompt_repro, max_tokens=60, temperatura=1.0)
    respostas_2.append(resp)
    print(f"  Tentativa {i+1}: {resp}")

iguais = all(r == respostas_2[0] for r in respostas_2)
print(f"\n  Todas idênticas? {'✅ Sim' if iguais else '❌ Não'}")

# Experimento 3: temperature=2, fixando semente no início
print("\n═══ Experimento 3: temperature=2, fixando semente no início ═══\n")
fixar_semente(42)
respostas_3 = []
for i in range(3):
    resp = gerar_resposta(prompt_repro, max_tokens=60, temperatura=2.0)
    respostas_3.append(resp)
    print(f"  Tentativa {i+1}: {resp}")

iguais = all(r == respostas_3[0] for r in respostas_3)
print(f"\n  Todas idênticas? {'✅ Sim' if iguais else '❌ Não'}")


### 8.5 Alucinações
As **alucinações** são respostas que parecem coerentes, mas contêm informações **inventadas, falsas ou não fundamentadas**.

Alguns tipos:
- **Invenção:** fatos, datas, números, nomes próprios
- **Excesso de confiança (Sobreconfiança):** afirmações sem evidência apresentadas como fatos
- **Citações falsas:** referências fabricadas a artigos científicos, livros, notícias ou outras fontes inexistentes

### 8.6 Por que aparecem?
Os modelos geram texto prevendo o token **mais provável**, não verificando a verdade. Um modelo "alucina" porque, estatisticamente, a sequência de palavras é provável, embora seja falsa. Mesmo que um modelo seja treinado completamente com informações verdadeiras, ele pode gerar alucinações. Existem técnicas que permitem diminuir a frequência das alucinações, mas elas não as eliminam.


In [ ]:
# ═══ 8.6-8.7 Exemplos e detecção de alucinações ═══

prompts_alucinacao = [
    "Quem escreveu o livro 'Fundamentos de Redes Neurais Quânticas' publicado em 2018?",
    "Quantas pessoas participaram da primeira conferência internacional de Small Language Models em Tóquio em 2020?",
    "Cite um paper acadêmico sobre otimização de modelos de linguagem pequenos com autores e ano.",
]

print("🔍 Explorando alucinações do modelo\n")

for prompt in prompts_alucinacao:
    resposta = gerar_resposta(prompt, max_tokens=120, temperatura=0.3)
    print(f"📝 Pergunta: {prompt}")
    print(f"🤖 Resposta: {resposta}")
    print(f"   ⚠️ VERIFICAR: É factual ou inventado?")
    print("-" * 60)
    print()

print("💡 Estratégias de mitigação:")
print("   • Pedir evidência explícita ('cite a fonte')")
print("   • Restringir formato ('responda somente se souber, senão diga não sei')")
print("   • Fornecer contexto no prompt (RAG)")
print("   • Abster-se se não souber: instruir o modelo a dizer 'não tenho essa informação'")


### **8.7 Mini exercício**

🧪 **Experimente isso:** Explorar as estratégias de mitigação de alucinações:
- Pedir evidência: "Você pode citar a fonte dessa informação?"
- Restringir formato: "Me dê apenas uma lista de fatos verificáveis." ou "Responda somente com 'Sim' ou 'Não'."
- Usar contexto fornecido: "Com base no seguinte texto, responda a pergunta..."
- Abster-se se não souber: "Se não tiver certeza, diga que não sabe."


### 8.8 Definir critério de avaliação por equipe

Perguntas para refletir:
- O que querem otimizar no seu projeto? (precisão, recall, clareza...)
- Que tipo de erro seria mais grave? (falso positivo, falso negativo, alucinação...)
- Como medirão se o modelo é "suficientemente bom"?

> 🧪 **Experimente isso:** Defina 3 critérios de avaliação para seu projeto.


---
## 9. Dataset mínimo para fine-tuning

### Formato instruction tuning
Para fine-tuning, precisamos de pares de **(instrução, resposta)** em formato de chat.

### Tamanho recomendado
- Mínimo: 10-20 exemplos (para experimentar)
- Ideal: 50-500 exemplos (para resultados notáveis)
- Os exemplos devem ser **consistentes**, **diversos** e **de alta qualidade**

### Cuidado com dados ruidosos
Exemplos contraditórios ou de baixa qualidade podem piorar o modelo.


In [ ]:
# ═══ 9.2 Criar dataset de instruções ═══
from datasets import Dataset

dados_treinamento = [
    {"messages": [
        {"role": "user", "content": "O que é aprendizado por transferência?"},
        {"role": "assistant", "content": "O aprendizado por transferência é uma técnica onde um modelo treinado em uma tarefa é reutilizado como ponto de partida para outra tarefa relacionada. Isso economiza tempo e dados, porque o modelo já aprendeu representações úteis que pode aproveitar no novo domínio."}
    ]},
    {"messages": [
        {"role": "user", "content": "Como funciona a quantização de modelos?"},
        {"role": "assistant", "content": "A quantização reduz a precisão numérica dos pesos do modelo, por exemplo de 32 bits para 8 ou 4 bits. Isso reduz o uso de memória e acelera a inferência com uma perda mínima de qualidade, tornando possível rodar modelos grandes em hardware limitado."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é uma rede neural convolucional?"},
        {"role": "assistant", "content": "Uma rede neural convolucional (CNN) é um tipo de rede especializada em processar dados com estrutura de grade, como imagens. Usa filtros que detectam padrões locais como bordas, texturas e formas, combinando-os em camadas sucessivas para reconhecer objetos complexos."}
    ]},
    {"messages": [
        {"role": "user", "content": "Qual é a diferença entre um modelo aberto e um fechado?"},
        {"role": "assistant", "content": "Um modelo aberto tem seus pesos disponíveis publicamente para download, inspeção e modificação. Um modelo fechado é acessado apenas via API sem ver seus parâmetros internos. Os modelos abertos permitem maior controle, privacidade e personalização."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é processamento de linguagem natural?"},
        {"role": "assistant", "content": "O processamento de linguagem natural (NLP) é uma área da inteligência artificial que trata da interação entre computadores e linguagem humana. Inclui tarefas como tradução, resumo, classificação de texto e geração de respostas."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é tokenização em modelos de linguagem?"},
        {"role": "assistant", "content": "A tokenização é o processo de dividir texto em unidades menores chamadas tokens, que podem ser palavras, subpalavras ou caracteres. O modelo trabalha com esses tokens em vez do texto bruto. Diferentes tokenizadores produzem fragmentações diferentes."}
    ]},
    {"messages": [
        {"role": "user", "content": "Por que é importante rodar modelos localmente?"},
        {"role": "assistant", "content": "Rodar modelos localmente garante privacidade porque os dados não são enviados a terceiros, permite personalização completa, reduz custos recorrentes e proporciona independência tecnológica ao não depender de serviços comerciais externos."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é o fine-tuning de um modelo?"},
        {"role": "assistant", "content": "O fine-tuning é o processo de continuar treinando um modelo pré-treinado com dados específicos de uma tarefa ou domínio particular. Isso adapta o modelo geral a necessidades concretas, melhorando seu desempenho nesse contexto sem treinar do zero."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é LoRA e para que serve?"},
        {"role": "assistant", "content": "LoRA (Low-Rank Adaptation) é uma técnica de fine-tuning eficiente que congela os pesos originais do modelo e adiciona matrizes treináveis de baixo rank. Isso reduz drasticamente a memória necessária e permite adaptar modelos grandes com hardware limitado."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é uma janela de contexto?"},
        {"role": "assistant", "content": "A janela de contexto é a quantidade máxima de tokens que um modelo pode processar em uma única entrada. Se o texto ultrapassar esse limite, o modelo não pode ver a informação que fica fora. Modelos típicos têm janelas de 2048 a 128000 tokens."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que são alucinações em modelos de linguagem?"},
        {"role": "assistant", "content": "As alucinações são respostas que parecem coerentes mas contêm informações inventadas ou falsas. Ocorrem porque o modelo gera texto baseando-se em probabilidades estatísticas, não em verificação de fatos. É importante sempre validar as saídas."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é um embedding em inteligência artificial?"},
        {"role": "assistant", "content": "Um embedding é uma representação numérica densa de dados (texto, imagens, etc.) em um espaço vetorial. Permite que conceitos similares fiquem próximos nesse espaço, facilitando que os modelos capturem relações semânticas."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é o formato GGUF?"},
        {"role": "assistant", "content": "GGUF é um formato de arquivo para armazenar modelos de linguagem quantizados, otimizado para inferência eficiente em CPU. É o formato usado pelo llama.cpp e permite rodar modelos grandes em hardware com recursos limitados."}
    ]},
    {"messages": [
        {"role": "user", "content": "Qual é a diferença entre inferência e treinamento?"},
        {"role": "assistant", "content": "O treinamento é o processo de ajustar os pesos do modelo usando dados, o que é computacionalmente custoso. A inferência é usar o modelo já treinado para gerar previsões ou respostas, o que é muito mais rápido e requer menos recursos."}
    ]},
    {"messages": [
        {"role": "user", "content": "O que é um hiperparâmetro?"},
        {"role": "assistant", "content": "Um hiperparâmetro é uma configuração definida antes do treinamento e que não é aprendida dos dados. Exemplos incluem a taxa de aprendizado, o tamanho do batch e o número de épocas. Sua escolha afeta significativamente o desempenho do modelo."}
    ]},
]

print(f"📊 Dataset criado com {len(dados_treinamento)} exemplos")
print(f"\nExemplo:")
print(f"  User: {dados_treinamento[0]['messages'][0]['content']}")
print(f"  Assistant: {dados_treinamento[0]['messages'][1]['content']}")


In [ ]:
# ═══ 9.3 Formatar e limpar dataset ═══

# Formatar usando o chat template do modelo
textos_formatados = []

for exemplo in dados_treinamento:
    texto = tokenizador.apply_chat_template(
        exemplo["messages"], tokenize=False, add_generation_prompt=False
    )
    textos_formatados.append({"text": texto})

conjunto_treinamento = Dataset.from_list(textos_formatados)

print(f"✅ Dataset formatado: {len(conjunto_treinamento)} exemplos")
print(f"\n📝 Exemplo formatado (primeiros 300 caracteres):")
print(conjunto_treinamento[0]["text"][:300])
print("\n🔍 Notaram algo curioso? O que é o system prompt que aparece no início de cada exemplo e por que está lá?")

# Verificar comprimentos
comprimentos = [len(tokenizador.encode(ex["text"])) for ex in conjunto_treinamento]

print(f"\n📊 Estatísticas de comprimento:")
print(f"   Mínimo: {min(comprimentos)} tokens")
print(f"   Máximo: {max(comprimentos)} tokens")
print(f"   Médio: {np.mean(comprimentos):.0f} tokens")

# Verificar consistência básica
print(f"\n🔍 Verificação de qualidade:")
for i, exemplo in enumerate(dados_treinamento):
    msgs = exemplo["messages"]
    if len(msgs) != 2:
        print(f"   ⚠️ Exemplo {i}: tem {len(msgs)} mensagens (esperado: 2)")
    elif not msgs[1]["content"].strip():
        print(f"   ⚠️ Exemplo {i}: resposta vazia")
print("   ✅ Verificação concluída")


---
## 10. Fine-tuning progressivo (Hugging Face + PEFT)

### 10.1 Supervised Fine-Tuning (SFT)
O **SFT** modifica **todos** os parâmetros do modelo. É potente, mas:
- Requer muita memória (todos os gradientes + optimizer states)
- Em um modelo de 1,7B parâmetros, precisa de ~14GB+ de RAM
- É prático apenas em GPU com VRAM suficiente

> ⚠️ Em CPU, os métodos **eficientes** (LoRA, Prompt Tuning) são preferíveis.


In [ ]:
# ═══ 10.1 SFT (demonstração — requer GPU ou MPS) ═══

if tem_cuda or tem_mps:
    from trl import SFTTrainer, SFTConfig

    # Recarregar modelo fresco para SFT
    modelo_sft = AutoModelForCausalLM.from_pretrained(
        nome_modelo, torch_dtype="auto", device_map="auto"
    )
    # Imprimir quantidade de parâmetros treináveis
    total_params = sum(p.numel() for p in modelo_sft.parameters())
    trainable_params = sum(p.numel() for p in modelo_sft.parameters() if p.requires_grad)
    print(f"Total de parâmetros: {total_params}")
    print(f"Parâmetros treináveis: {trainable_params}")

    config_sft = SFTConfig(
        output_dir="./resultados_sft",
        max_steps=2,
        per_device_train_batch_size=1,
        learning_rate=2e-5,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        dataset_text_field="text",
        max_length=256,
    )

    treinador_sft = SFTTrainer(
        model=modelo_sft,
        processing_class=tokenizador,
        args=config_sft,
        train_dataset=conjunto_treinamento,
    )

    print("⏳ Treinando SFT (2 passos de demonstração)...")
    treinador_sft.train()
    print("✅ SFT concluído")

    del treinador_sft, modelo_sft
    gc.collect()
    if tem_cuda:
        torch.cuda.empty_cache()
else:
    print("ℹ️  SFT completo requer GPU ou MPS (modifica todos os parâmetros).")
    print("   Em CPU, usaremos LoRA e Prompt Tuning que são muito mais eficientes.")
    print("   → Avançar para seção 10.2 (LoRA)")


### 10.2 LoRA (Low-Rank Adaptation)

**LoRA** congela os pesos originais e adiciona **matrizes de baixo rank** treináveis a camadas específicas. Vantagens:
- Treina apenas ~1-2% dos parâmetros totais
- Muito menos memória e mais rápido
- O modelo base não é modificado: adaptadores podem ser adicionados/removidos
- Funciona em CPU (mais lento, mas viável)


In [ ]:
# ═══ 10.2.2 LoRA — Implementação ═══
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("⏳ Preparando modelo para LoRA...")

# Carregar modelo fresco
modelo_para_lora = AutoModelForCausalLM.from_pretrained(
    nome_modelo, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
modelo_para_lora.train()

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

modelo_lora = get_peft_model(modelo_para_lora, config_lora)
modelo_lora.print_trainable_parameters()

# Configurar treinamento
config_treinamento_lora = SFTConfig(
    output_dir="./resultados_lora",
    max_steps=3,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=256,
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    gradient_accumulation_steps=1,
    optim="adamw_torch",
)

treinador_lora = SFTTrainer(
    model=modelo_lora,
    processing_class=tokenizador,
    args=config_treinamento_lora,
    train_dataset=conjunto_treinamento,
)

print("\n⏳ Treinando com LoRA (3 passos)...")
print("   (em CPU pode demorar alguns minutos por passo)\n")
resultado_lora = treinador_lora.train()
print(f"\n✅ LoRA concluído. Loss final: {resultado_lora.training_loss:.4f}")


In [ ]:
# ═══ 10.2.3-10.2.4 Avaliação e comparação LoRA ═══

print("📊 Comparação: modelo base vs modelo com LoRA\n")

prompts_comparacao = [
    "O que é aprendizado por transferência?",
    "Como funciona a quantização de modelos?",
    "O que é uma rede neural convolucional?",
]
respostas_antes_ft = {}

# Gerar com LoRA (desativar/ativar adaptadores)
modelo_lora.eval()

for prompt in prompts_comparacao:
    # Com LoRA
    modelo_lora.enable_adapter_layers()
    resp_lora = gerar_resposta(prompt, max_tokens=100, temperatura=0.0, modelo_usar=modelo_lora)

    # Sem LoRA (base)
    modelo_lora.disable_adapter_layers()
    resp_base = gerar_resposta(prompt, max_tokens=100, temperatura=0.0, modelo_usar=modelo_lora)
    respostas_antes_ft[prompt] = resp_base
    modelo_lora.enable_adapter_layers()

    print(f"📝 Prompt: {prompt}")
    print(f"   BASE:  {resp_base}")
    print(f"   LoRA:  {resp_lora}")
    print()

print("💡 Com apenas 3 passos e 15 exemplos, as mudanças são mínimas.")
print("   Para resultados notáveis, usar 50+ exemplos e 100+ passos.")

# Salvar adaptadores LoRA
caminho_lora = "./adaptadores_lora"
modelo_lora.save_pretrained(caminho_lora)
print(f"\n💾 Adaptadores LoRA salvos em {caminho_lora}")

# Limpar
del treinador_lora, modelo_lora, modelo_para_lora
gc.collect()


### 10.3 Prompt Tuning

**Prompt Tuning** adiciona "tokens virtuais" treináveis no início do input. É o método mais leve:
- Treina apenas os embeddings dos tokens virtuais
- Parâmetros treináveis: ~20 embeddings × dimensão do modelo
- Extremamente rápido e com uso mínimo de memória


In [ ]:
# ═══ 10.3.2 Prompt Tuning — Implementação ═══
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model

print("⏳ Preparando modelo para Prompt Tuning...")

modelo_para_pt = AutoModelForCausalLM.from_pretrained(
    nome_modelo, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
modelo_para_pt.train()

config_prompt_tuning = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20,
    prompt_tuning_init=PromptTuningInit.RANDOM,
)

modelo_pt = get_peft_model(modelo_para_pt, config_prompt_tuning)
modelo_pt.print_trainable_parameters()

config_treinamento_pt = SFTConfig(
    output_dir="./resultados_prompt_tuning",
    max_steps=5,
    per_device_train_batch_size=1,
    learning_rate=3e-2,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=256,
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
)

treinador_pt = SFTTrainer(
    model=modelo_pt,
    processing_class=tokenizador,
    args=config_treinamento_pt,
    train_dataset=conjunto_treinamento,
)

print("\n⏳ Treinando Prompt Tuning (5 passos)...\n")
resultado_pt = treinador_pt.train()
print(f"\n✅ Prompt Tuning concluído. Loss final (média móvel, pode diferir do último passo): {resultado_pt.training_loss:.4f}")

# Limpar
del treinador_pt, modelo_pt, modelo_para_pt
_ = gc.collect()


---
## 11. Avaliação pós fine-tuning

### Antes vs depois
Comparamos as respostas do modelo base com as do modelo fine-tuneado com LoRA.


In [ ]:
# ═══ 11.1-11.3 Comparação antes vs depois (LoRA) ═══
from peft import PeftModel

print("⏳ Carregando modelo base + adaptadores LoRA...\n")
modelo_base = AutoModelForCausalLM.from_pretrained(
    nome_modelo, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
modelo_avaliacao = PeftModel.from_pretrained(modelo_base, "./adaptadores_lora")
modelo_avaliacao.eval()

print("📊 Comparação ANTES vs DEPOIS do fine-tuning com LoRA\n")

for prompt in prompts_comparacao:
    resp_antes = respostas_antes_ft.get(prompt, "(não disponível)")
    resp_depois = gerar_resposta(prompt, max_tokens=100, temperatura=0.0, modelo_usar=modelo_avaliacao)

    print(f"📝 {prompt}")
    print(f"   ANTES:   {resp_antes[:200]}")
    print(f"   DEPOIS: {resp_depois[:200]}")
    print()

del modelo_avaliacao, modelo_base
gc.collect()


### Discussão prática

**Quando vale a pena fazer fine-tuning?**
- Quando você precisa de um comportamento muito específico e consistente
- Quando tem dados de qualidade para seu domínio
- Quando o prompting não é suficiente

**Quando o prompting é suficiente?**
- Para tarefas gerais ou exploratórias
- Quando não tem dados rotulados
- Para protótipos rápidos

> 🧪 **Reflexão:** Seu projeto se beneficiaria de fine-tuning ou o prompting é suficiente?


---
## 12. Aplicação a projetos

### 12.1 Definir problema
Defina claramente:
1. **Tarefa:** o que o modelo deve fazer? (classificar, resumir, extrair, gerar...)
2. **Input:** o que ele recebe? (texto livre, formulário, documento...)
3. **Output:** o que deve produzir? (rótulo, resumo, resposta estruturada...)
4. **Critério de sucesso:** como saber se funciona bem?

### 12.2 Escolher abordagem

| Abordagem | Quando usar |
|-----------|------------|
| **Fine-tuning** | |
| Prompting | Protótipo rápido, sem dados rotulados |
| SFT | Muitos dados, GPU disponível, máxima personalização |
| LoRA | Tem 50+ exemplos, precisa de especialização |
| Prompt Tuning | Quer a forma mais leve, com poucos exemplos |
| **Inferência** ||
| Ollama | Deploy simples em CPU, sem código Python complexo |
| llama.cpp | Máxima eficiência em CPU, controle fino |


In [ ]:
# ═══ 12.3 Template de implementação rápida ═══

print("🚀 Template para aplicar ao seu projeto\n")

# --- Configuração do projeto ---
NOME_PROJETO = "Meu Projeto"  # 🧪 Mude isso
TAREFA = "classificacao"  # classificacao | resumo | extracao | geracao
PROMPT_SISTEMA = "Você é um assistente útil que responde de forma clara e concisa."

# --- Template de prompt ---
modelos_prompt = {
    "classificacao": (
        "{sistema}\n\n"
        "Classifique o seguinte texto em uma destas categorias: {categorias}.\n"
        "Responda SOMENTE com o nome da categoria.\n\n"
        "Texto: {texto}\n\nCategoria:"
    ),
    "resumo": (
        "{sistema}\n\n"
        "Resuma o seguinte texto em 2-3 frases.\n\n"
        "Texto: {texto}\n\nResumo:"
    ),
    "extracao": (
        "{sistema}\n\n"
        "Extraia as seguintes informações do texto: {campos}.\n"
        "Responda em formato JSON.\n\n"
        "Texto: {texto}\n\nInformações extraídas:"
    ),
    "geracao": (
        "{sistema}\n\n"
        "{instrucao}\n\n"
        "Contexto: {texto}\n\nResposta:"
    ),
}

# --- Exemplo de uso ---
modelo_prompt = modelos_prompt.get(TAREFA, modelos_prompt["geracao"])

# Recarregar modelo base para a demo
modelo_app = AutoModelForCausalLM.from_pretrained(
    nome_modelo, torch_dtype="auto", low_cpu_mem_usage=True
)

exemplo_texto = "Uma nova biblioteca de código aberto permite executar modelos de linguagem de 3 bilhões de parâmetros em celulares com apenas 2GB de RAM."

prompt_final = modelo_prompt.format(
    sistema=PROMPT_SISTEMA,
    categorias="tecnologia, ciência, educação, economia",
    texto=exemplo_texto,
    campos="tema, tecnologia_mencionada, requisito_tecnico",
    instrucao="Analise o impacto tecnológico desta notícia.",
)

resposta_app = gerar_resposta(prompt_final, max_tokens=100, modelo_usar=modelo_app)
print(f"📋 Projeto: {NOME_PROJETO}")
print(f"📝 Tarefa: {TAREFA}")
print(f"📄 Texto: {exemplo_texto}")
print(f"\n🤖 Resposta:")
print(resposta_app)

del modelo_app
gc.collect()

print("\n\n### 12.4 Próximo experimento")
print("1. Adapte o prompt ao seu caso real")
print("2. Teste com 5-10 exemplos do seu projeto")
print("3. Avalie os resultados com seus critérios (seção 8)")
print("4. Decida se precisa de fine-tuning ou se o prompting é suficiente")


---
## 13. Encerramento

### 13.1 Síntese

Neste workshop exploramos:
1. **Setup e hardware** — Como configurar um ambiente local para SLMs
2. **Inferência** — Gerar texto com Hugging Face, Ollama e llama.cpp
3. **Eficiência** — Quantização e otimização para hardware limitado
4. **Tokenização** — Desigualdades entre idiomas e fertility rate
5. **Avaliação** — Métricas, reprodutibilidade e alucinações
6. **Fine-tuning** — SFT, LoRA e Prompt Tuning
7. **Aplicação** — Templates para projetos reais

### 13.2 Próximos passos segundo caminho

| Caminho | Próximo passo |
|---------|--------------|
| 🟢 Simples | Testar Ollama com dados do seu projeto |
| 🟡 Experimentação | Analisar tokenização nos seus dados, comparar parâmetros |
| 🔴 Profundidade | Criar dataset de 50+ exemplos, fazer LoRA completo |
| ⚙️ Hardware limitado | Explorar modelos GGUF menores, otimizar llama.cpp |

### Recursos
- [Hugging Face Hub](https://huggingface.co/) — Modelos e datasets
- [PEFT](https://github.com/huggingface/peft) — Fine-tuning eficiente
- [Ollama](https://ollama.ai/) — Modelos locais fáceis
- [llama.cpp](https://llama-cpp.com/) — Execução eficiente em CPU

---

**Obrigada por participar! 🎉**

*Rede Feminista em IA para a América Latina e o Caribe — Eixo de Inovação*
